# Sentiment model experiments — automated T4 pipeline

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Approve the Google Drive prompt. Everything else is automatic. If Colab disconnects,
just **Run all** again — finished experiments are skipped (results live in Drive).

What it does, in order:
1. Installs pinned dependencies and writes the project package (bundle SHA-256 `80978ba762f5906b…`).
2. Downloads TweetEval sentiment (official Cardiff NLP files) and the CrowdFlower GOP-debate tweets; re-runs the data audit.
3. **Reference (validation only):** your current model `suhailabdulwasi/ai4all-distilroberta-sentiment` and Cardiff's off-the-shelf sentiment model.
4. **Stage 0/1:** reproduction of the original recipe + 6-architecture screen.
5. **Stage 2:** learning-rate / class-weight sweep for the top-2 architectures (generated from stage-1 *validation* results).
6. **Stage 3:** large models for the top base architectures. **Stage 4:** extra seeds of the best configuration.
7. **Model selection is frozen** (validation macro-F1, pre-declared rule) and written to `selection.json`.
8. **Only then**, one evaluation on the TweetEval test set and the GOP election benchmark (full + high-agreement subset), with 95% bootstrap CIs, calibration and error analysis. A lock file prevents re-testing a different selection.

Integrity rules enforced in code: test labels are never loaded during training/selection; early stopping and selection use validation only; the GOP data is never used for training or tuning.
Your existing notebook, Hugging Face model and GitHub `main` are **not** modified.

Expected runtime on a T4: ~3.5–5 h with all stages (≈1.5 h if `RUN_LARGE=False` and `RUN_SEEDS=False`). Peak Drive usage ≈ 4–6 GB (checkpoints are pruned after each stage).

In [ ]:
# ---- Settings -------------------------------------------------------------
RUN_LARGE = True            # stage 3: large models (~1.5 h extra)
RUN_SEEDS = True            # stage 4: 2 extra seeds of the best config
KEEP_TOP_CHECKPOINTS = 4    # best checkpoint kept per architecture, top-N architectures
DRIVE_ROOT = "/content/drive/MyDrive/sentiment-experiments"
CURRENT_MODEL = "suhailabdulwasi/ai4all-distilroberta-sentiment"

In [ ]:
import subprocess, sys, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *'transformers==5.22.1 datasets==5.1.0 accelerate==1.15.0 sentencepiece protobuf emoji==2.14.1 tabulate pyyaml scikit-learn scipy matplotlib'.split()], check=True)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, json, hashlib, pathlib
for sub in ("runs", "reports", "data"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)
RUN_ROOT, REPORT_DIR, DATA_DIR = f"{DRIVE_ROOT}/runs", f"{DRIVE_ROOT}/reports", f"{DRIVE_ROOT}/data"

In [ ]:
# Project source: xz-compressed JSON of src/, configs/, scripts/ (readable copy lives in the GitHub repo).
# Generated by scripts/build_colab_notebook.py. The SHA-256 check fails loudly if this cell is altered.
import base64, lzma
BUNDLE_B64 = "..."
BUNDLE_SHA256 = "80978ba762f5906b80c3aed974a12327e9b2a34d2ba48bd104bc352d22652432"
BUNDLE = json.loads(lzma.decompress(base64.b64decode(BUNDLE_B64)))
assert hashlib.sha256(json.dumps(BUNDLE, sort_keys=True).encode()).hexdigest() == BUNDLE_SHA256, "corrupted bundle"
PKG = pathlib.Path("/content/sentiment_pkg")
for rel, text in BUNDLE.items():
    p = PKG / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text)
sys.path.insert(0, str(PKG / "src"))
pathlib.Path(REPORT_DIR, "code_bundle_sha256.txt").write_text(BUNDLE_SHA256)
print(f"Wrote {len(BUNDLE)} files, bundle {BUNDLE_SHA256[:16]}")

In [ ]:
# Data download + audit (duplicates, leakage, distribution shift, GOP label checks)
import urllib.request
from sentiment.data import download_tweeteval, GOP_URL, GOP_SHA256
TE_ROOT = download_tweeteval(DATA_DIR)
GOP_PATH = pathlib.Path(DATA_DIR, "gop", "Sentiment.csv")
GOP_PATH.parent.mkdir(parents=True, exist_ok=True)
if not GOP_PATH.exists():
    urllib.request.urlretrieve(GOP_URL, GOP_PATH)
assert hashlib.sha256(GOP_PATH.read_bytes()).hexdigest() == GOP_SHA256, "GOP file changed upstream"
subprocess.run([sys.executable, str(PKG / "scripts/audit_data.py"), "--data-dir", DATA_DIR,
                "--gop", str(GOP_PATH), "--out", f"{REPORT_DIR}/data_audit.json"],
               check=True, env={**os.environ, "PYTHONPATH": str(PKG / "src")}, stdout=subprocess.DEVNULL)
audit = json.load(open(f"{REPORT_DIR}/data_audit.json"))
print("Sizes OK:", audit["tweeteval"]["sizes_match_expected"])
print("Cross-split overlap (normalised):", audit["tweeteval"]["cross_split_normalized"])
print("Label shift train vs test (Cramer's V):", audit["tweeteval"]["distribution_shift"]["train_vs_test"]["label_cramers_v"])

In [ ]:
# Reference models — VALIDATION ONLY
from sentiment.pipeline import reference_eval
ref_path = pathlib.Path(REPORT_DIR, "reference_validation.json")
if not ref_path.exists():
    display(reference_eval([CURRENT_MODEL, "cardiffnlp/twitter-roberta-base-sentiment-latest"], TE_ROOT, ref_path))
else:
    print(json.dumps({k: {m: round(v[m], 4) for m in ("accuracy", "macro_f1")} for k, v in json.load(open(ref_path)).items()}, indent=1))

In [ ]:
# Stage 0 + 1: reproduction and architecture screen
from sentiment.config import load_experiments
from sentiment.pipeline import run_stage, prune_checkpoints
from sentiment.selection import load_results, comparison_table
run_stage(load_experiments(PKG / "configs/experiments.yaml"), TE_ROOT, RUN_ROOT)
print("pruned:", prune_checkpoints(RUN_ROOT, keep=KEEP_TOP_CHECKPOINTS))
display(comparison_table(load_results(RUN_ROOT)).drop(columns=["hypothesis"]))

In [ ]:
# Stage 2: LR + class-weight sweep for the top-2 screened architectures (validation-driven)
from sentiment.plan import lr_sweep
run_stage(lr_sweep(load_results(RUN_ROOT)), TE_ROOT, RUN_ROOT)
print("pruned:", prune_checkpoints(RUN_ROOT, keep=KEEP_TOP_CHECKPOINTS))

In [ ]:
# Stage 3: large models
from sentiment.plan import large_models
if RUN_LARGE:
    run_stage(large_models(load_results(RUN_ROOT)), TE_ROOT, RUN_ROOT)
    print("pruned:", prune_checkpoints(RUN_ROOT, keep=KEEP_TOP_CHECKPOINTS))

In [ ]:
# Stage 4: seed repeats of the best configuration
from sentiment.plan import seed_repeats
if RUN_SEEDS:
    run_stage(seed_repeats(load_results(RUN_ROOT)), TE_ROOT, RUN_ROOT)
    print("pruned:", prune_checkpoints(RUN_ROOT, keep=KEEP_TOP_CHECKPOINTS))

## Freeze model selection (validation only)
Rule (declared before any test evaluation): rank runs by validation macro-F1; consider
averaging the top-k distinct architectures (k = 1…4); accept a larger ensemble only if it
improves validation macro-F1 by ≥ 0.5 pt **and** the paired-bootstrap 95% CI excludes 0.

In [ ]:
from sentiment.pipeline import select
from sentiment.selection import seed_summary
selection = select(RUN_ROOT, TE_ROOT, REPORT_DIR)
print(json.dumps({k: selection[k] for k in ("members", "val_macro_f1", "val_accuracy")}, indent=1))
display(seed_summary(comparison_table(load_results(RUN_ROOT))))

## One-time held-out evaluation
Runs once on the frozen selection: TweetEval test, GOP full (de-duplicated) and GOP high-agreement
subset (confidence = 1.0, threshold fixed in advance). The original model is evaluated alongside
for a paired comparison. A lock file refuses a second test evaluation of a *different* selection.

In [ ]:
from sentiment.pipeline import final_evaluation
final = final_evaluation(REPORT_DIR, TE_ROOT, GOP_PATH, baseline_model=CURRENT_MODEL)
for name in ("tweeteval_test", "gop_full_dedup", "gop_conf1"):
    r = final[name]; b = r["baseline"]
    print(f"{name:16s} n={r['n']:5d} | NEW acc {r['accuracy']:.4f} {[round(x,4) for x in r['accuracy_ci95']]} "
          f"macroF1 {r['macro_f1']:.4f} {[round(x,4) for x in r['macro_f1_ci95']]} | "
          f"ORIGINAL acc {b['accuracy']:.4f} macroF1 {b['macro_f1']:.4f} | "
          f"dF1 {r['paired_vs_baseline_macro_f1']['diff']:+.4f} CI {[round(x,4) for x in r['paired_vs_baseline_macro_f1']['ci95']]}")

In [ ]:
# Error analysis + figures
import pandas as pd
from IPython.display import Image
print(pd.read_csv(f"{REPORT_DIR}/confusion_pairs_tweeteval_test.csv").to_string(index=False))
print(pd.read_csv(f"{REPORT_DIR}/slices_tweeteval_test.csv").to_string(index=False))
for f in ("model_comparison", "learning_curves", "confusion_tweeteval_test", "reliability_tweeteval_test", "confusion_gop_conf1"):
    display(Image(f"{REPORT_DIR}/figures/{f}.png"))

In [ ]:
# Package lightweight results (no model weights) for review
import shutil
for d in pathlib.Path(RUN_ROOT).iterdir():
    for f in ("result.json", "error.txt"):
        if (d / f).exists():
            dst = pathlib.Path(REPORT_DIR, "runs", d.name); dst.mkdir(parents=True, exist_ok=True)
            shutil.copy(d / f, dst / f)
print(shutil.make_archive(f"{DRIVE_ROOT}/results_bundle", "zip", REPORT_DIR))
print("Done. Best checkpoints are in", RUN_ROOT, "(not uploaded anywhere).")